# Regressão Logística de Churn — Seguradora Xperiun  *(v2)*
**MBA Xperiun · Data Challenge 4**

Modelo **explicativo** de churn gerenciável (cancelamento voluntário).

## Mudanças v2 em relação à v1
| Tema | Mudança |
|------|---------|
| Erros-padrão | Clusterizados por `cliente_id` no statsmodels (`cov_type='cluster'`) |
| Contagens de portfólio | Calculadas **cronologicamente** (até `data_inicio` de cada apólice) |
| Features novas | `ratio_premio`, `mes_inicio`, `trimestre_inicio`, `tem_auto/vida/residencial`, `tem_cross_sell` |
| Gênero PJ | Corrigido para `'Não Informado'` quando `tipo_cliente = 'Pessoa Jurídica'` |
| Leakage removido | `cac_perdido_ajustado` e `taxa_adimplencia` excluídos |
| Desbalanceamento | `freq_weights` no statsmodels (mantém interpretabilidade dos betas) |
| Estratificação | `iterative-stratification` (multilabel) |
| Categorias raras | Agrupadas em `'Outros'` antes do encoding |
| Nota de unidade | Apólice como unidade; erros clusterizados corrigem dependência intra-cliente |


## Pré-requisito: Ambiente Virtual

Execute **uma única vez** no terminal antes de abrir o Jupyter:

```bash
# 1. Criar o ambiente virtual na pasta do projeto
python -m venv .venv_churn

# 2. Ativar (Windows PowerShell)
.venv_churn\Scripts\Activate.ps1
# Ativar (macOS / Linux)
source .venv_churn/bin/activate

# 3. Instalar dependências
pip install statsmodels scikit-learn pandas numpy matplotlib seaborn scipy \
            openpyxl iterative-stratification

# 4. Registrar como kernel do Jupyter
pip install ipykernel
python -m ipykernel install --user --name=venv_churn --display-name='Python (churn)'

# 5. Abrir o Jupyter apontando para este kernel
jupyter notebook
```

> **VSCode:** selecione o kernel `Python (churn)` na barra superior do notebook.


## Etapa 1 · Imports e Configuração

In [ ]:
import sqlite3, warnings, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ks_2samp, chi2_contingency, mannwhitneyu

# Statsmodels — inferência (betas, OR, IC 95%, p-valores clusterizados)
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Scikit-learn — métricas, CV, L1
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    confusion_matrix, classification_report, roc_curve
)

# Estratificação multilabel
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
np.random.seed(SEED)

print('✅ Bibliotecas carregadas.')


## Etapa 2 · Query SQL

### Decisões de design da query

**Contagens de portfólio cronológicas:** `qtd_apolices_cliente`, `qtd_ramos_cliente` e 
`qtd_coberturas_cliente` contam **todas** as apólices do cliente (incluindo suspensas, 
falecimentos e venda do bem) com `data_inicio < data_inicio_da_apolice_atual`. 
Isso evita leakage: cada apólice enxerga apenas o histórico anterior do cliente.

**Ratio de prêmio:** `ratio_premio = premio_mensal / premio_medio_mensal` — quanto 
o cliente paga vs. a média do produto. Valor > 1 indica prêmio acima da média.

**Sazonalidade:** `mes_inicio` e `trimestre_inicio` extraídos de `data_inicio`.

**Dummies de ramo:** `tem_auto`, `tem_vida`, `tem_residencial`, `tem_cross_sell` 
— calculados historicamente (mesmo critério das contagens).

**Leakage evitado:** `status`, `motivo_cancelamento`, `data_cancelamento`, 
`receita_total`, `parcelas_pagas`, `cac_perdido`, `taxa_adimplencia`, 
`dias_ate_churn`, `janela_retencao` — **todos excluídos**.


In [ ]:
QUERY_SQL = """
-- =========================================================================
-- Base de Modelagem — Regressão Logística de Churn
-- Seguradora Xperiun · Data Challenge 4
-- Unidade de análise: APÓLICE
-- Erros-padrão serão clusterizados por cliente_id no Python.
-- =========================================================================

-- Etapa A: Universo base com variável resposta e filtros
WITH universo AS (

 SELECT f_apolice.apolice_id
      , f_apolice.cliente_id
      , f_apolice.produto_id
      , f_apolice.canal_id
      , f_apolice.data_inicio
      , f_apolice.vigencia_meses
      , f_apolice.premio_mensal
      , f_apolice.receita_esperada
      , f_apolice.comissao
      , f_apolice.forma_pagamento
      -- Variável resposta: churn GERENCIÁVEL
      , CASE
            WHEN f_apolice.status = 'Cancelada'
             AND f_apolice.motivo_cancelamento NOT IN (
                 'Falecimento do segurado'
                ,'Venda do bem segurado'
             )
            THEN 1
            ELSE 0
        END AS indicador_churn
   FROM f_apolice
  WHERE f_apolice.status != 'Suspensa'
    -- Exclui cancelamentos estruturais do conjunto
    AND NOT (
            f_apolice.status = 'Cancelada'
        AND f_apolice.motivo_cancelamento IN (
            'Falecimento do segurado'
           ,'Venda do bem segurado'
        )
    )
    -- Exclui datas inconsistentes
    AND (
            f_apolice.data_cancelamento IS NULL
         OR f_apolice.data_cancelamento >= f_apolice.data_inicio
    )
    -- Exclui apólices muito recentes (exposição insuficiente)
    AND f_apolice.data_inicio < '2024-10-01'
)

-- Etapa B: Portfólio cronológico por cliente
-- Conta TODAS as apólices do cliente com data_inicio < data da apólice atual.
-- Inclui suspensas, falecimentos, venda do bem — pois representavam
-- o histórico real do cliente no momento da emissão.
, portfolio_historico AS (

 SELECT universo.apolice_id
      , universo.cliente_id
      , universo.data_inicio
      -- Quantidade de apólices anteriores (histórico completo)
      , (
            SELECT COUNT(f_apolice.apolice_id)
              FROM f_apolice
             WHERE f_apolice.cliente_id   = universo.cliente_id
               AND f_apolice.data_inicio  < universo.data_inicio
        ) AS qtd_apolices_anteriores
      -- Ramos distintos em apólices anteriores
      , (
            SELECT COUNT(DISTINCT d_produto.ramo)
              FROM f_apolice
              JOIN d_produto ON f_apolice.produto_id = d_produto.produto_id
             WHERE f_apolice.cliente_id  = universo.cliente_id
               AND f_apolice.data_inicio < universo.data_inicio
        ) AS qtd_ramos_anteriores
      -- Coberturas distintas em apólices anteriores
      , (
            SELECT COUNT(DISTINCT d_produto.cobertura)
              FROM f_apolice
              JOIN d_produto ON f_apolice.produto_id = d_produto.produto_id
             WHERE f_apolice.cliente_id  = universo.cliente_id
               AND f_apolice.data_inicio < universo.data_inicio
        ) AS qtd_coberturas_anteriores
      -- Dummy: já tinha apólice de Automóvel antes desta
      , (
            SELECT COUNT(f_apolice.apolice_id)
              FROM f_apolice
              JOIN d_produto ON f_apolice.produto_id = d_produto.produto_id
             WHERE f_apolice.cliente_id  = universo.cliente_id
               AND f_apolice.data_inicio < universo.data_inicio
               AND d_produto.ramo        = 'Automóvel'
        ) > 0 AS tinha_auto
      -- Dummy: já tinha apólice de Vida antes desta
      , (
            SELECT COUNT(f_apolice.apolice_id)
              FROM f_apolice
              JOIN d_produto ON f_apolice.produto_id = d_produto.produto_id
             WHERE f_apolice.cliente_id  = universo.cliente_id
               AND f_apolice.data_inicio < universo.data_inicio
               AND d_produto.ramo        = 'Vida'
        ) > 0 AS tinha_vida
      -- Dummy: já tinha apólice Residencial antes desta
      , (
            SELECT COUNT(f_apolice.apolice_id)
              FROM f_apolice
              JOIN d_produto ON f_apolice.produto_id = d_produto.produto_id
             WHERE f_apolice.cliente_id  = universo.cliente_id
               AND f_apolice.data_inicio < universo.data_inicio
               AND d_produto.ramo        = 'Residencial'
        ) > 0 AS tinha_residencial
   FROM universo
)

-- Query principal
 SELECT universo.apolice_id
      , universo.cliente_id
      -- === VARIÁVEL RESPOSTA ===
      , universo.indicador_churn
      -- === FEATURES DA APÓLICE ===
      , universo.vigencia_meses
      , universo.premio_mensal
      , universo.receita_esperada
      , universo.comissao
      , universo.forma_pagamento
      -- Sazonalidade: mês e trimestre de início
      , CAST(strftime('%m', universo.data_inicio) AS INTEGER) AS mes_inicio
      , CASE
            WHEN CAST(strftime('%m', universo.data_inicio) AS INTEGER) <= 3  THEN 1
            WHEN CAST(strftime('%m', universo.data_inicio) AS INTEGER) <= 6  THEN 2
            WHEN CAST(strftime('%m', universo.data_inicio) AS INTEGER) <= 9  THEN 3
            ELSE 4
        END AS trimestre_inicio
      -- === FEATURES DO CLIENTE ===
      , d_cliente.tipo_cliente
      -- Gênero: 'Não Informado' para Pessoa Jurídica (PJ não tem gênero)
      , CASE
            WHEN d_cliente.tipo_cliente = 'Pessoa Jurídica' THEN 'Não Informado'
            ELSE d_cliente.genero
        END AS genero
      , d_cliente.tempo_cliente_meses
      , CASE
            WHEN d_cliente.faixa_etaria = '18-25' THEN '1) 18-25'
            WHEN d_cliente.faixa_etaria = '26-35' THEN '2) 26-35'
            WHEN d_cliente.faixa_etaria = '36-45' THEN '3) 36-45'
            WHEN d_cliente.faixa_etaria = '46-55' THEN '4) 46-55'
            WHEN d_cliente.faixa_etaria = '56-65' THEN '5) 56-65'
            ELSE '6) 65+'
        END AS faixa_etaria
      , CASE
            WHEN d_cliente.tempo_cliente_meses <= 12                         THEN '1) 0-12 meses'
            WHEN d_cliente.tempo_cliente_meses BETWEEN 13 AND 48             THEN '2) 13-48 meses'
            ELSE '3) >48 meses'
        END AS faixa_tempo_casa
      -- === FEATURES GEOGRÁFICAS ===
      , d_regiao.regiao
      -- === FEATURES DO PRODUTO ===
      , d_produto.ramo
      , d_produto.cobertura
      , d_produto.premio_medio_mensal
      , d_produto.franquia_media
      -- Razão: prêmio do cliente vs. prêmio médio do produto
      , ROUND(universo.premio_mensal / NULLIF(d_produto.premio_medio_mensal, 0), 4) AS ratio_premio
      -- === FEATURES DO CANAL ===
      , d_canal.tipo                         AS tipo_canal
      , d_canal.nome_canal
      , d_canal.comissao_percentual / 100.0  AS comissao_percentual
      -- === PORTFÓLIO CRONOLÓGICO DO CLIENTE ===
      , portfolio_historico.qtd_apolices_anteriores
      , portfolio_historico.qtd_ramos_anteriores
      , portfolio_historico.qtd_coberturas_anteriores
      , CAST(portfolio_historico.tinha_auto        AS INTEGER) AS tinha_auto
      , CAST(portfolio_historico.tinha_vida        AS INTEGER) AS tinha_vida
      , CAST(portfolio_historico.tinha_residencial AS INTEGER) AS tinha_residencial
      -- Cross-sell: cliente já tinha apólice de pelo menos 2 ramos distintos
      , CASE
            WHEN portfolio_historico.qtd_ramos_anteriores >= 2 THEN 1
            ELSE 0
        END AS tinha_cross_sell
   FROM universo
   JOIN d_cliente          ON universo.cliente_id  = d_cliente.cliente_id
   JOIN d_regiao           ON d_cliente.regiao_id  = d_regiao.regiao_id
   JOIN d_produto          ON universo.produto_id  = d_produto.produto_id
   JOIN d_canal            ON universo.canal_id    = d_canal.canal_id
   JOIN portfolio_historico ON universo.apolice_id = portfolio_historico.apolice_id
  ORDER BY universo.apolice_id;
"""

print('Query SQL v2 pronta.')
print('Opção A: cole no DB Browser for SQLite → exporte CSV.')
print('Opção B: ajuste CAMINHO_DB e execute na célula seguinte.')


## Etapa 3 · Carregamento e Exploração Inicial

In [ ]:
# ── Opção A: direto via Python ──────────────────────────────────────────────
CAMINHO_DB = r'AJUSTE_AQUI/seguradora_xperiun.db'   # <── ajuste aqui

conn   = sqlite3.connect(CAMINHO_DB)
df_raw = pd.read_sql_query(QUERY_SQL, conn)
conn.close()

# ── Opção B: CSV exportado ───────────────────────────────────────────────────
# CAMINHO_CSV = r'AJUSTE_AQUI/base_modelo.csv'
# df_raw = pd.read_csv(CAMINHO_CSV, sep=';', decimal=',')

print(f'✅ {df_raw.shape[0]:,} linhas | {df_raw.shape[1]} colunas')

# ── Distribuição da variável resposta ───────────────────────────────────────
TAXA_CHURN_GLOBAL = df_raw['indicador_churn'].mean()
dist = df_raw['indicador_churn'].value_counts()
pct  = df_raw['indicador_churn'].value_counts(normalize=True)
print(f'\nTaxa global de churn gerenciável: {TAXA_CHURN_GLOBAL:.4%}')
print(pd.DataFrame({'Rótulo':{0:'Não-Churn',1:'Churn'}, 'Qtd':dist,
                    '%':pct.map('{:.2%}'.format)}))

# ── Verificação do ajuste de gênero PJ ──────────────────────────────────────
pj_com_genero = df_raw[
    (df_raw['tipo_cliente']=='Pessoa Jurídica') &
    (~df_raw['genero'].isin(['Não Informado', None, '']))
].shape[0]
print(f'\n✅ PJ com gênero diferente de Não Informado: {pj_com_genero} (deve ser 0)')

# ── Tipos e nulos ────────────────────────────────────────────────────────────
print('\n', df_raw.dtypes.value_counts().to_string())
nulos = df_raw.isnull().sum()
if nulos.sum() > 0:
    print('\nColunas com nulos:')
    print(nulos[nulos > 0].to_string())
else:
    print('\n✅ Sem valores nulos.')


## Etapa 4 · Feature Engineering Complementar

Pequenos ajustes que não cabem na query SQL:
- `mes_inicio` e `trimestre_inicio` como **categóricas** (efeito sazonal não linear)
- Fusão de categorias raras em `'Outros'` (< `LIMIAR_RARO`% da base)


In [ ]:
df = df_raw.copy()

# ── Tratar mes_inicio e trimestre_inicio como categóricas ───────────────────
df['mes_inicio']       = df['mes_inicio'].astype(str).str.zfill(2)  # '01'..'12'
df['trimestre_inicio'] = 'T' + df['trimestre_inicio'].astype(str)    # 'T1'..'T4'

# ── Fusão de categorias raras ────────────────────────────────────────────────
LIMIAR_RARO = 0.01   # categorias com < 1% da base → agrupadas em 'Outros'

COLUNAS_CAT = [
    'forma_pagamento', 'tipo_cliente', 'genero', 'faixa_etaria',
    'faixa_tempo_casa', 'regiao', 'ramo', 'cobertura',
    'tipo_canal', 'nome_canal',
    'mes_inicio', 'trimestre_inicio'
]

MAPEAMENTO_RAROS = {}
for col in COLUNAS_CAT:
    freq = df[col].value_counts(normalize=True)
    raras = freq[freq < LIMIAR_RARO].index.tolist()
    if raras:
        MAPEAMENTO_RAROS[col] = raras
        df[col] = df[col].where(~df[col].isin(raras), other='Outros')
        print(f'  {col}: {len(raras)} categoria(s) rara(s) → "Outros": {raras}')

if not MAPEAMENTO_RAROS:
    print('✅ Nenhuma categoria rara encontrada.')

print(f'\n✅ DataFrame final: {df.shape[0]:,} linhas | {df.shape[1]} colunas')


## Etapa 5 · Definição das Colunas do Modelo

In [ ]:
# Identificadores — nunca entram no modelo
COLUNAS_ID     = ['apolice_id', 'cliente_id']
COLUNAS_TARGET = ['indicador_churn']

# Categóricas: todas serão encodadas com categoria de referência dinâmica
COLUNAS_CAT_MODELO = [
    'forma_pagamento',     # 4 valores
    'tipo_cliente',        # PF / PJ
    'genero',              # M / F / Não Informado
    'faixa_etaria',        # 6 faixas ordenadas
    'faixa_tempo_casa',    # 3 faixas — testada em paralelo com tempo_cliente_meses
    'regiao',              # 5 regiões
    'ramo',                # Auto / Vida / Residencial
    'cobertura',           # Básica / Completa / Premium
    'tipo_canal',          # 4 tipos (agrupador de nome_canal)
    'nome_canal',          # canal específico — alta cardinalidade; VIF/seleção decide
    'mes_inicio',          # sazonalidade mensal
    'trimestre_inicio',    # sazonalidade trimestral
]

# Numéricas contínuas
COLUNAS_NUM_MODELO = [
    'vigencia_meses',            # prazo contratado
    'premio_mensal',             # prêmio do cliente
    'receita_esperada',          # premio_mensal × vigencia_meses (colinear — VIF decide)
    'comissao',                  # comissão paga (colinear — VIF decide)
    'tempo_cliente_meses',       # contínua (padrão) — mais informação que a faixa
    'comissao_percentual',       # percentual do canal
    'premio_medio_mensal',       # média do produto (colinear com ratio_premio — VIF decide)
    'franquia_media',            # franquia média do produto
    'ratio_premio',              # premio_mensal / premio_medio_mensal
    'qtd_apolices_anteriores',   # portfólio histórico
    'qtd_ramos_anteriores',      # diversidade de ramos histórica
    'qtd_coberturas_anteriores', # diversidade de coberturas histórica
]

# Binárias (já 0/1 — não precisam de encoding)
COLUNAS_BIN_MODELO = [
    'tinha_auto',
    'tinha_vida',
    'tinha_residencial',
    'tinha_cross_sell',
]

print('Colunas candidatas ao modelo:')
print(f'  Categóricas: {len(COLUNAS_CAT_MODELO)}')
print(f'  Numéricas:   {len(COLUNAS_NUM_MODELO)}')
print(f'  Binárias:    {len(COLUNAS_BIN_MODELO)}')
print(f'  Total bruto: {len(COLUNAS_CAT_MODELO)+len(COLUNAS_NUM_MODELO)+len(COLUNAS_BIN_MODELO)}')
print('\n(Após dummies, o número de colunas será maior.)')


## Etapa 6 · Split Estratificado 70/30 — MultilabelStratifiedShuffleSplit

`iterative-stratification` equilibra múltiplas variáveis simultaneamente, 
mesmo quando há estratos pequenos.


In [ ]:
# Criar rótulos multilabel para estratificação
# Ordem de prioridade: churn, faixa_tempo_casa, faixa_etaria, ramo, tipo_canal
COLUNAS_ESTRATO = ['indicador_churn', 'faixa_tempo_casa', 'faixa_etaria', 'ramo', 'tipo_canal']

# Converter cada coluna de estrato para int (multilabel precisa de matriz numérica)
from sklearn.preprocessing import LabelEncoder

Y_estrato = np.column_stack([
    LabelEncoder().fit_transform(df[col].astype(str))
    for col in COLUNAS_ESTRATO
])

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.30, random_state=SEED
)

for idx_treino, idx_teste in msss.split(df, Y_estrato):
    df_treino = df.iloc[idx_treino].copy().reset_index(drop=True)
    df_teste  = df.iloc[idx_teste].copy().reset_index(drop=True)

churn_tr = df_treino['indicador_churn'].mean()
churn_te = df_teste['indicador_churn'].mean()

print(f'✅ Split estratificado (multilabel):')
print(f'   Treino: {len(df_treino):,} ({len(df_treino)/len(df):.1%})  |  Churn: {churn_tr:.4%}')
print(f'   Teste:  {len(df_teste):,}  ({len(df_teste)/len(df):.1%})  |  Churn: {churn_te:.4%}')
print(f'   Global: {TAXA_CHURN_GLOBAL:.4%}')
print('\n→ As três taxas de churn devem estar muito próximas.')


## Etapa 7 · Encoding com Categoria de Referência Dinâmica

Referência = categoria cuja taxa de churn é mais próxima da média global (~16%) 
**e** com volume ≥ 1% da base de treino.


In [ ]:
def escolher_referencia(df, col, target, min_pct=0.01):
    """
    Retorna a categoria mais próxima da taxa global de churn
    com volume suficiente (>= min_pct da base).
    """
    taxa_global   = df[target].mean()
    churn_por_cat = df.groupby(col)[target].mean()
    vol_por_cat   = df[col].value_counts(normalize=True)
    candidatas    = (churn_por_cat - taxa_global).abs().sort_values()
    for cat in candidatas.index:
        if vol_por_cat.get(cat, 0) >= min_pct:
            return cat
    return candidatas.index[0]  # fallback sem restrição


def encodar_dummies(df_treino, df_teste, colunas_cat, target='indicador_churn'):
    """
    Gera dummies removendo a categoria de referência calculada no treino.
    df_teste recebe apenas as colunas geradas no treino (segurança).
    Retorna (df_treino_enc, df_teste_enc, dict_referencias).
    """
    referencias = {}
    tr, te = df_treino.copy(), df_teste.copy()

    print(f'\nTaxa global de churn (treino): {df_treino[target].mean():.4%}')
    print(f'{"Coluna":<22} {"Referência":<30} {"Churn ref":>10} {"Volume ref":>12}')
    print('-' * 78)

    for col in colunas_cat:
        if col not in tr.columns:
            continue
        ref = escolher_referencia(df_treino, col, target)
        referencias[col] = ref
        churn_ref  = df_treino.groupby(col)[target].mean().get(ref, np.nan)
        vol_ref    = df_treino[col].value_counts().get(ref, 0)
        print(f'  {col:<20} {str(ref):<30} {churn_ref:>10.4%} {vol_ref:>12,}')

        # Gerar dummies e remover referência
        dum_tr = pd.get_dummies(tr[col], prefix=col, dtype=int)
        dum_te = pd.get_dummies(te[col], prefix=col, dtype=int)
        col_ref = f'{col}_{ref}'
        dum_tr  = dum_tr.drop(columns=[col_ref], errors='ignore')
        dum_te  = dum_te.reindex(columns=dum_tr.columns, fill_value=0)
        tr = pd.concat([tr.drop(columns=[col]), dum_tr], axis=1)
        te = pd.concat([te.drop(columns=[col]), dum_te], axis=1)

    return tr, te, referencias


df_treino_enc, df_teste_enc, REFERENCIAS = encodar_dummies(
    df_treino, df_teste, COLUNAS_CAT_MODELO
)

# Montar X e y
EXCLUIR = set(COLUNAS_ID + COLUNAS_TARGET + ['faixa_tempo_casa'])
# faixa_tempo_casa: excluir do modelo principal (tempo_cliente_meses já está contínua)
# Mantemos as dummies dela para comparação, mas não no X principal

COLUNAS_MODELO = [
    c for c in df_treino_enc.columns
    if c not in EXCLUIR
    and not c.startswith('faixa_tempo_casa_')   # exclui dummies da faixa
]

X_treino = df_treino_enc[COLUNAS_MODELO].astype(float)
y_treino = df_treino_enc['indicador_churn'].astype(int)
X_teste  = df_teste_enc[COLUNAS_MODELO].astype(float)
y_teste  = df_teste_enc['indicador_churn'].astype(int)
GRUPO_TREINO = df_treino_enc['cliente_id'].values  # para erros clusterizados

print(f'\n✅ X_treino: {X_treino.shape}  |  X_teste: {X_teste.shape}')
print(f'   Variáveis candidatas: {len(COLUNAS_MODELO)}')


## Etapa 8 · Análise Bivariada

In [ ]:
resultados_biv = []

print(f"{'='*70}\nANÁLISE BIVARIADA (Base de Treino)")
print(f"Taxa global de churn: {TAXA_CHURN_GLOBAL:.4%}\n{'='*70}")

# Categóricas (originais, antes do encoding)
for col in COLUNAS_CAT_MODELO:
    if col not in df_treino.columns:
        continue
    tab = pd.crosstab(df_treino[col], df_treino['indicador_churn'])
    chi2, p, dof, _ = chi2_contingency(tab)
    cr = df_treino.groupby(col)['indicador_churn'].mean().sort_values(ascending=False)
    vol = df_treino[col].value_counts()
    resultados_biv.append({'variavel':col,'tipo':'cat','estatistica':chi2,'p_valor':p,
                           'amplitude':cr.max()-cr.min()})
    sig = '***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else 'ns'))
    print(f'\n📊 {col.upper()} — χ²={chi2:.1f}  p={p:.4f} {sig}')
    r = cr.rename('Churn%').to_frame()
    r['Churn%'] = r['Churn%'].map('{:.2%}'.format)
    r['Vol'] = vol; r['%Base'] = (vol/len(df_treino)).map('{:.1%}'.format)
    print(r.to_string())

# Numéricas
NUM_BIV = COLUNAS_NUM_MODELO + COLUNAS_BIN_MODELO
for col in NUM_BIV:
    if col not in df_treino.columns:
        continue
    g0 = df_treino.loc[df_treino['indicador_churn']==0, col].dropna()
    g1 = df_treino.loc[df_treino['indicador_churn']==1, col].dropna()
    stat, p = mannwhitneyu(g0, g1, alternative='two-sided')
    resultados_biv.append({'variavel':col,'tipo':'num','estatistica':stat,'p_valor':p,'amplitude':np.nan})
    sig = '***' if p<.001 else ('**' if p<.01 else ('*' if p<.05 else 'ns'))
    print(f'\n📈 {col.upper()} — U={stat:.0f}  p={p:.4f} {sig}')
    print(f'   Não-churn: μ={g0.mean():.4f}  med={g0.median():.4f}')
    print(f'   Churn:     μ={g1.mean():.4f}  med={g1.median():.4f}')

df_bivariada = pd.DataFrame(resultados_biv).sort_values('p_valor')
print(f"\n{'='*70}\nRESUMO — {len(df_bivariada)} variáveis testadas")
print(df_bivariada[['variavel','tipo','estatistica','p_valor']].to_string(index=False))


## Etapa 9 · VIF — Multicolinearidade

In [ ]:
X_vif = sm.add_constant(X_treino.fillna(0))

print('Calculando VIF...')
vif_data = pd.DataFrame({
    'variavel': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values.astype(float), i)
            for i in range(X_vif.shape[1])]
}).query("variavel != 'const'").sort_values('VIF', ascending=False)

print(f"{'='*55}\nVIF (menor = menos colinear)\n{'='*55}")
print(vif_data.to_string(index=False))

# Classificação por nível de risco
vif_data['risco'] = pd.cut(vif_data['VIF'], bins=[0,5,10,np.inf],
                            labels=['✅ OK','⚠️ Moderado','❌ Grave'])
print(f"\nResumo por risco:")
print(vif_data['risco'].value_counts().to_string())

VARS_VIF_OK = vif_data[vif_data['VIF'] <= 10]['variavel'].tolist()
VARS_VIF_GRAVE = vif_data[vif_data['VIF'] > 10]['variavel'].tolist()
if VARS_VIF_GRAVE:
    print(f'\n⚠️  Variáveis com VIF > 10 (remover ou monitorar):')
    print(vif_data[vif_data['VIF']>10][['variavel','VIF']].to_string(index=False))


## Etapa 10 · Seleção por Regularização L1 (Lasso)

In [ ]:
scaler = StandardScaler()
X_treino_sc = pd.DataFrame(scaler.fit_transform(X_treino.fillna(0)), columns=X_treino.columns)
X_teste_sc  = pd.DataFrame(scaler.transform(X_teste.fillna(0)),      columns=X_teste.columns)

print('Ajustando L1 com CV para encontrar C ótimo...')
lasso_cv = LogisticRegressionCV(
    cv=5, penalty='l1', solver='saga',
    class_weight='balanced',
    Cs=np.logspace(-4, 4, 30),
    random_state=SEED, max_iter=5000, n_jobs=-1
)
lasso_cv.fit(X_treino_sc, y_treino)

coef_lasso = pd.DataFrame({'variavel': X_treino.columns,
                            'coef_l1':  lasso_cv.coef_[0]})\
               .sort_values('coef_l1', key=abs, ascending=False)

VARS_L1 = coef_lasso[coef_lasso['coef_l1'] != 0]['variavel'].tolist()
print(f'\nC ótimo: {lasso_cv.C_[0]:.6f}')
print(f'L1 selecionou {len(VARS_L1)} de {len(X_treino.columns)} variáveis')
print(coef_lasso[coef_lasso['coef_l1'] != 0].to_string(index=False))


## Etapa 11 · Seleção Stepwise Forward-Backward

In [ ]:
def stepwise_logistico(X, y, p_entrada=0.05, p_saida=0.10, verbose=True):
    """
    Seleção Stepwise Forward-Backward para Regressão Logística.
    Usa statsmodels.Logit. Critério: p-valor do coeficiente.
    """
    incluidas  = []
    candidatas = list(X.columns)
    iteracao   = 0

    while True:
        iteracao += 1
        melhorou = False

        # Forward: incluir melhor candidata
        p_cand = {}
        for var in [v for v in candidatas if v not in incluidas]:
            tent = incluidas + [var]
            Xt   = sm.add_constant(X[tent].fillna(0), has_constant='add')
            try:
                res = sm.Logit(y, Xt).fit(disp=0, maxiter=200)
                p_cand[var] = res.pvalues.get(var, 1.0)
            except Exception:
                p_cand[var] = 1.0

        if p_cand:
            melhor = min(p_cand, key=p_cand.get)
            if p_cand[melhor] < p_entrada:
                incluidas.append(melhor)
                melhorou = True
                if verbose:
                    print(f'  [{iteracao:03d}] + Incluindo "{melhor}" (p={p_cand[melhor]:.4f})')

        # Backward: remover pior
        if len(incluidas) > 1:
            Xa  = sm.add_constant(X[incluidas].fillna(0), has_constant='add')
            res = sm.Logit(y, Xa).fit(disp=0, maxiter=200)
            pv  = res.pvalues.drop('const', errors='ignore')
            pior = pv.idxmax()
            if pv[pior] > p_saida:
                incluidas.remove(pior)
                melhorou = True
                if verbose:
                    print(f'  [{iteracao:03d}] - Removendo "{pior}" (p={pv[pior]:.4f})')

        if not melhorou:
            break

    return incluidas


print('Executando Stepwise (pode demorar alguns minutos)...\n')
VARS_STEPWISE = stepwise_logistico(X_treino, y_treino)
print(f'\n✅ Stepwise: {len(VARS_STEPWISE)} variáveis')
print(VARS_STEPWISE)


## Etapa 12 · AIC / BIC

In [ ]:
def aic_bic(X, y, vars_):
    Xf = sm.add_constant(X[vars_].fillna(0), has_constant='add')
    r  = sm.Logit(y, Xf).fit(disp=0, maxiter=300)
    return r.aic, r.bic, len(vars_)

resultados_aic = []
for nome, vs in [('Full', list(X_treino.columns)), ('Stepwise', VARS_STEPWISE), ('L1', VARS_L1)]:
    if vs:
        a, b, n = aic_bic(X_treino, y_treino, vs)
        resultados_aic.append({'Modelo': nome, 'AIC': a, 'BIC': b, 'Vars': n})

df_aic = pd.DataFrame(resultados_aic)
print(f"{'='*55}\nCOMPARAÇÃO AIC / BIC (menor = melhor)\n{'='*55}")
print(df_aic.to_string(index=False))


## Etapa 13 · Consenso de Seleção

In [ ]:
consensus = pd.DataFrame({'variavel': list(X_treino.columns)})
consensus['l1']       = consensus['variavel'].isin(VARS_L1).astype(int)
consensus['stepwise'] = consensus['variavel'].isin(VARS_STEPWISE).astype(int)
consensus['vif_ok']   = consensus['variavel'].isin(VARS_VIF_OK).astype(int)
consensus['votos']    = consensus['l1'] + consensus['stepwise']
consensus = consensus.sort_values(['votos','stepwise','l1'], ascending=False)

print(f"{'='*65}\nCONSENSO — L1 + Stepwise + VIF\n{'='*65}")
print(consensus.to_string(index=False))

# Variáveis finais: ao menos 1 método + VIF OK
VARS_FINAIS = consensus[
    (consensus['votos'] >= 1) & (consensus['vif_ok'] == 1)
]['variavel'].tolist()

if not VARS_FINAIS:
    VARS_FINAIS = [v for v in VARS_STEPWISE if v in VARS_VIF_OK]
    print('⚠️  Fallback para Stepwise ∩ VIF_OK')

print(f'\n✅ Variáveis para o modelo final: {len(VARS_FINAIS)}')
print(VARS_FINAIS)


## Etapa 14 · Modelo Final — statsmodels.Logit

### Dois ajustes para dados desbalanceados e dependência intra-cluster:

1. **`freq_weights`** — pesos amostrais que amplificam os churns (~16%) para equilibrar 
as classes **mantendo a interpretabilidade dos betas e odds ratios**. 
Equivale a `class_weight='balanced'` do sklearn sem perder a escala original dos coeficientes.

2. **`cov_type='cluster', cov_kwds={'groups': grupo}`** — erros-padrão clusterizados 
por `cliente_id`, corrigindo a violação de independência (um cliente pode ter múltiplas apólices).


In [ ]:
# ── Pesos amostrais para balancear classes ───────────────────────────────────
# Peso inversamente proporcional à frequência de cada classe no treino.
# w(não-churn) = n_total / (2 × n_não-churn) | w(churn) = n_total / (2 × n_churn)
n_total  = len(y_treino)
n_churn  = y_treino.sum()
n_nchurn = n_total - n_churn

peso_churn  = n_total / (2 * n_churn)
peso_nchurn = n_total / (2 * n_nchurn)

pesos = np.where(y_treino == 1, peso_churn, peso_nchurn)

print(f'Peso não-churn: {peso_nchurn:.4f}')
print(f'Peso churn:     {peso_churn:.4f}')
print(f'Razão:          {peso_churn/peso_nchurn:.2f}×')

# ── Ajustar modelo ───────────────────────────────────────────────────────────
X_treino_sm = sm.add_constant(X_treino[VARS_FINAIS].fillna(0), has_constant='add')

print('\nAjustando statsmodels.Logit com freq_weights e erros clusterizados...')

modelo_base = sm.Logit(y_treino, X_treino_sm, freq_weights=pesos)
modelo_final = modelo_base.fit(
    disp=0, maxiter=500,
    cov_type='cluster',
    cov_kwds={'groups': GRUPO_TREINO}
)

print(modelo_final.summary2())
print(f'\nAIC:                  {modelo_final.aic:.2f}')
print(f'BIC:                  {modelo_final.bic:.2f}')
print(f'Pseudo R² (McFadden): {modelo_final.prsquared:.4f}')
print(f'\n📌 Erros-padrão clusterizados por cliente_id.')
print(f'   Os betas e odds ratios NÃO são afetados pela clusterização —')
print(f'   apenas os p-valores e IC 95% são corrigidos.')


### tModeloRegressao — Extração da Tabela de Coeficientes

In [ ]:
coef      = modelo_final.params
ic_95     = modelo_final.conf_int(alpha=0.05)
p_valores = modelo_final.pvalues
or_vals   = np.exp(coef)
or_ic_inf = np.exp(ic_95.iloc[:, 0])
or_ic_sup = np.exp(ic_95.iloc[:, 1])

df_resultado = pd.DataFrame({
    'variavel':       coef.index,
    'coeficiente_b':  coef.values,
    'odds_ratio':     or_vals.values,
    'p_valor':        p_valores.values,
    'ic_95_inf_or':   or_ic_inf.values,
    'ic_95_sup_or':   or_ic_sup.values,
    'significante':   (p_valores < 0.05).values,
    'direcao':        np.where(or_vals > 1, '↑ Aumenta churn', '↓ Protege contra churn')
})
df_resultado = (
    df_resultado[df_resultado['variavel'] != 'const']
    .sort_values('odds_ratio', ascending=False)
    .reset_index(drop=True)
)

print(f"{'='*90}\ntModeloRegressao\n{'='*90}")
print(df_resultado.to_string(index=False))
print(f"\nVariáveis significantes (p<0.05): {df_resultado['significante'].sum()}")
print(f"OR > 1 (aumentam churn): {(df_resultado['odds_ratio']>1).sum()}")
print(f"OR < 1 (protegem):       {(df_resultado['odds_ratio']<1).sum()}")


## Etapa 15 · Cross-Validation k=10 (sklearn)

In [ ]:
clf_cv = LogisticRegression(
    penalty=None,
    class_weight='balanced',
    solver='lbfgs',
    max_iter=5000,
    random_state=SEED
)
pipeline_cv = Pipeline([('sc', StandardScaler()), ('lr', clf_cv)])
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

res_cv = cross_validate(
    pipeline_cv, X_treino[VARS_FINAIS].fillna(0), y_treino,
    cv=cv10,
    scoring={'auc':'roc_auc','logloss':'neg_log_loss','brier':'neg_brier_score'},
    return_train_score=True, n_jobs=-1
)

print(f"{'='*62}\nCROSS-VALIDATION k=10\n{'='*62}")
print(f"{'Métrica':<20} {'CV Média':>10} {'CV DP':>8} {'Treino':>10}")
print('-'*52)
for lbl, ck, tk in [('AUC-ROC','test_auc','train_auc'),
                     ('Log-Loss','test_logloss','train_logloss'),
                     ('Brier','test_brier','train_brier')]:
    cv_v = np.abs(res_cv[ck]);  tr_v = np.abs(res_cv[tk])
    print(f'  {lbl:<18} {cv_v.mean():>10.4f} {cv_v.std():>8.4f} {tr_v.mean():>10.4f}')
print('\n→ CV ≈ Treino = boa generalização.')


## Etapa 16 · Aplicação na Base de Teste

In [ ]:
X_treino_pred = sm.add_constant(X_treino[VARS_FINAIS].fillna(0), has_constant='add')
X_teste_pred  = sm.add_constant(X_teste[VARS_FINAIS].fillna(0),  has_constant='add')

prob_treino = modelo_final.predict(X_treino_pred)
prob_teste  = modelo_final.predict(X_teste_pred)

THRESHOLD = TAXA_CHURN_GLOBAL
pred_treino = (prob_treino >= THRESHOLD).astype(int)
pred_teste  = (prob_teste  >= THRESHOLD).astype(int)

df_teste_scored = df_teste.copy()
df_teste_scored['prob_churn']  = prob_teste.values
df_teste_scored['pred_churn']  = pred_teste
df_teste_scored['score_decil'] = pd.qcut(prob_teste, q=10, labels=False, duplicates='drop') + 1

print(f'Threshold: {THRESHOLD:.4%}')
print(f'Preditos como Churn: {pred_teste.sum():,} ({pred_teste.mean():.2%})')
print(f'Real Churn (teste):  {y_teste.sum():,} ({y_teste.mean():.2%})')


## Etapa 17 · Métricas de Performance

In [ ]:
def ks_stat(y_true, y_prob):
    g1 = y_prob[np.array(y_true)==1]
    g0 = y_prob[np.array(y_true)==0]
    return ks_2samp(g1, g0)

def relatorio(y_true, y_prob, y_pred, nome):
    auc = roc_auc_score(y_true, y_prob)
    ll  = log_loss(y_true, y_prob)
    br  = brier_score_loss(y_true, y_prob)
    ks, ksp = ks_stat(y_true, y_prob)
    cm  = confusion_matrix(y_true, y_pred)
    print(f"\n{'='*60}\nPERFORMANCE — {nome.upper()}\n{'='*60}")
    print(f'  AUC-ROC:     {auc:.4f}')
    print(f'  KS:          {ks:.4f}  (p={ksp:.4f})')
    print(f'  Log-Loss:    {ll:.4f}')
    print(f'  Brier Score: {br:.4f}')
    print(f'\n  Matriz de Confusão (threshold={THRESHOLD:.2%}):')
    print(f'              Pred 0    Pred 1')
    print(f'  Real 0: {cm[0,0]:10,} {cm[0,1]:9,}')
    print(f'  Real 1: {cm[1,0]:10,} {cm[1,1]:9,}')
    print(f'\n{classification_report(y_true, y_pred, target_names=["Não-Churn","Churn"], digits=4)}')
    return {'nome':nome,'auc':auc,'ks':ks,'logloss':ll,'brier':br}

met_tr = relatorio(y_treino, prob_treino, pred_treino, 'Treino')
met_te = relatorio(y_teste,  prob_teste,  pred_teste,  'Teste')

print(f"\n📊 Generalização:")
for m in ['auc','ks']:
    d = met_tr[m] - met_te[m]
    flag = '✅' if abs(d) <= 0.05 else '⚠️  Overfitting'
    print(f'  Δ{m.upper()} treino-teste: {d:+.4f}  {flag}')


## Etapa 18 · Curvas ROC / KS e Forest Plot (Odds Ratios)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Regressão Logística de Churn — Seguradora Xperiun', fontsize=13, fontweight='bold')

# ── ROC ──────────────────────────────────────────────────────────────────────
for yt, yp, nm, cor in [(y_treino,prob_treino,'Treino','#2E9CCA'),(y_teste,prob_teste,'Teste','#F0A500')]:
    fpr,tpr,_ = roc_curve(yt,yp)
    axes[0].plot(fpr,tpr,label=f'{nm} (AUC={roc_auc_score(yt,yp):.3f})',color=cor,lw=2)
axes[0].plot([0,1],[0,1],'--',color='gray',lw=1)
axes[0].set(xlabel='FPR',ylabel='TPR',title='Curva ROC')
axes[0].legend(); axes[0].set_aspect('equal')

# ── KS ───────────────────────────────────────────────────────────────────────
for yt, yp, nm, cor in [(y_treino,prob_treino,'Treino','#2E9CCA'),(y_teste,prob_teste,'Teste','#F0A500')]:
    thrs = np.linspace(0,1,300)
    tprl,fprl = [],[]
    for t in thrs:
        pr = (yp>=t).astype(int)
        c  = confusion_matrix(yt,pr,labels=[0,1]).ravel()
        tn,fp,fn,tp = c if len(c)==4 else (c[0],0,0,0)
        tprl.append(tp/(tp+fn+1e-10)); fprl.append(fp/(fp+tn+1e-10))
    ks_arr = np.array(tprl)-np.array(fprl)
    idx    = np.argmax(ks_arr)
    axes[1].plot(thrs,tprl,color=cor,lw=1.5,alpha=0.8)
    axes[1].plot(thrs,fprl,color=cor,lw=1.5,ls='--',alpha=0.8)
    axes[1].axvline(thrs[idx],color=cor,lw=0.8,ls=':')
    axes[1].annotate(f'KS={ks_arr[idx]:.3f} ({nm})',xy=(thrs[idx]+.01,.5),color=cor,fontsize=9)
axes[1].set(xlabel='Threshold',ylabel='Taxa',title='Curva KS')

# ── Forest Plot — Odds Ratios ─────────────────────────────────────────────────
df_forest = df_resultado[df_resultado['significante']].sort_values('odds_ratio')
y_pos = range(len(df_forest))
axes[2].barh(y_pos, df_forest['odds_ratio']-1,
             left=1,
             color=['#E74C3C' if o>1 else '#27AE60' for o in df_forest['odds_ratio']],
             alpha=0.7)
axes[2].errorbar(
    df_forest['odds_ratio'], y_pos,
    xerr=[df_forest['odds_ratio']-df_forest['ic_95_inf_or'],
          df_forest['ic_95_sup_or']-df_forest['odds_ratio']],
    fmt='none', color='#0D1F3C', capsize=3, lw=1
)
axes[2].axvline(1.0, color='#0D1F3C', lw=1.5, ls='--')
axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(df_forest['variavel'], fontsize=8)
axes[2].set(xlabel='Odds Ratio (IC 95%)', title='Forest Plot — vars. significantes')

plt.tight_layout()
plt.savefig('modelo_churn_xperiun.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo: modelo_churn_xperiun.png')


## Etapa 19 · Exportação

In [ ]:
# tModeloRegressao.csv — importar no Power BI
df_resultado.to_csv('tModeloRegressao.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
print('✅ tModeloRegressao.csv')

# Scores da base de teste
df_teste_scored[['apolice_id','cliente_id','indicador_churn',
                  'prob_churn','pred_churn','score_decil']]\
    .to_csv('scores_base_teste.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
print('✅ scores_base_teste.csv')

# Excel consolidado
df_metricas = pd.DataFrame([
    {'Base':'Treino','AUC':met_tr['auc'],'KS':met_tr['ks'],'LogLoss':met_tr['logloss'],'Brier':met_tr['brier']},
    {'Base':'Teste', 'AUC':met_te['auc'],'KS':met_te['ks'],'LogLoss':met_te['logloss'],'Brier':met_te['brier']},
    {'Base':'CV k=10 (média)','AUC':np.abs(res_cv['test_auc']).mean(),'KS':None,'LogLoss':None,'Brier':None},
])

with pd.ExcelWriter('resultados_regressao_logistica.xlsx', engine='openpyxl') as writer:
    df_resultado.to_excel(writer,     sheet_name='tModeloRegressao',  index=False)
    df_bivariada.to_excel(writer,     sheet_name='Bivariada',          index=False)
    vif_data.to_excel(writer,         sheet_name='VIF',                index=False)
    consensus.to_excel(writer,        sheet_name='Selecao_Variaveis',  index=False)
    df_metricas.to_excel(writer,      sheet_name='Metricas',           index=False)
    df_aic.to_excel(writer,           sheet_name='AIC_BIC',            index=False)
    df_teste_scored[['apolice_id','cliente_id','indicador_churn',
                      'prob_churn','pred_churn','score_decil']]\
        .to_excel(writer,             sheet_name='Scores_Teste',       index=False)

print('✅ resultados_regressao_logistica.xlsx')
print('\n📁 Arquivos gerados:')
print('   • tModeloRegressao.csv                → importar no Power BI')
print('   • scores_base_teste.csv               → probabilidades individuais')
print('   • resultados_regressao_logistica.xlsx → 7 abas consolidadas')
print('   • modelo_churn_xperiun.png            → ROC + KS + Forest Plot')


## Apêndice · Nota sobre Inversão da Unidade de Análise (Cliente)

Você perguntou se seria possível **inverter para o cliente como unidade**. 
É possível, mas há trade-offs importantes:

### Como fazer
Agregar `f_apolice` por `cliente_id`, criando uma linha por cliente com:
- `indicador_churn_cliente = MAX(indicador_churn)` — 1 se qualquer apólice churnou
- `qtd_apolices`, `qtd_ramos`, `qtd_coberturas` — sem necessidade de cronologia
- `premio_mensal_total = SUM(premio_mensal)` — receita total do cliente
- Features demográficas (únicas por cliente)

### Por que **não** inverter neste caso
| Aspecto | Apólice (atual) | Cliente (invertido) |
|---------|----------------|--------------------|
| **Registros** | ~117.500 | ~25.000 |
| **Granularidade** | Cada contrato é avaliado | Comportamento agregado |
| **Variável resposta** | 'Cancelou esta apólice?' | 'Churnou em alguma apólice?' |
| **Features de apólice** | Naturais (vigência, prêmio) | Precisam de agregação |
| **Pergunta de negócio** | 'Quais apólices têm risco?' | 'Quais clientes têm risco?' |
| **Aplicação no BI** | Score por apólice ativa | Score por cliente |

**Recomendação:** para **scoring de retenção** (aplicar no Power BI e decidir ação por apólice), 
a unidade apólice com erros clusterizados é o caminho certo. 
Para **segmentação de clientes** em campanhas de relacionamento, 
a inversão para cliente faz sentido como um segundo modelo complementar.
